In [ ]:
!pip install biopython torch_geometric torchmd-net
!wget -q https://files.rcsb.org/download/1QLX.pdb

In [ ]:
import os
import torch
from torch_geometric.data import InMemoryDataset, Data
from Bio import PDB
import glob

ELEMENT_TO_Z = {'H': 1, 'C': 6, 'N': 7, 'O': 8, 'S': 16, 'P': 15}

class PrionDataset(InMemoryDataset):
    def __init__(self, root, transform=None, pre_transform=None):
        super().__init__(root, transform, pre_transform)
        self.load(self.processed_paths[0])
        
    @property
    def raw_file_names(self):
        return glob.glob(os.path.join(self.raw_dir, '*.pdb'))
        
    @property
    def processed_file_names(self):
        return ['prion_data.pt']
        
    def download(self):
        pass
        
    def process(self):
        data_list = []
        parser = PDB.PDBParser(QUIET=True)
        raw_files = self.raw_file_names
        if not raw_files:
            print(f"No PDB files found in {self.raw_dir}.")
            return
        for raw_path in raw_files:
            print(f"Parsing {raw_path}...")
            structure = parser.get_structure("prion", raw_path)
            z_list = []
            pos_list = []
            for model in structure:
                for chain in model:
                    for residue in chain:
                        for atom in residue:
                            element = atom.element.strip().upper()
                            if element in ELEMENT_TO_Z:
                                z_list.append(ELEMENT_TO_Z[element])
                                pos_list.append(atom.coord)
                            else:
                                z_list.append(0)
                                pos_list.append(atom.coord)
            import numpy as np
            z_tensor = torch.tensor(z_list, dtype=torch.long)
            pos_tensor = torch.tensor(np.array(pos_list), dtype=torch.float32)
            data = Data(z=z_tensor, pos=pos_tensor)
            if self.pre_transform is not None:
                data = self.pre_transform(data)
            data_list.append(data)
        self.save(data_list, self.processed_paths[0])
        print(f"Successfully processed {len(data_list)} PDB structures.")


In [ ]:
import os
import torch
from torch_geometric.loader import DataLoader
from torchmdnet.models.model import create_model
from torchmdnet.scripts.train import get_args

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Testing Prion pipeline on: {device}")

raw_dir = os.path.join('data', 'prion', 'raw')
os.makedirs(raw_dir, exist_ok=True)
if os.path.exists('1QLX.pdb'):
    os.rename('1QLX.pdb', os.path.join(raw_dir, '1QLX.pdb'))
    
print("\nLoading Prion Dataset...")
dataset = PrionDataset(root='./data/prion')
data = dataset[0]
num_atoms = data.z.shape[0]
print(f"Structure 0 (1QLX) has {num_atoms} atoms.")

loader = DataLoader(dataset, batch_size=1, shuffle=False)
batch = next(iter(loader)).to(device)

import sys
original_argv = sys.argv
sys.argv = ['run_prion_simulation.py']
model_args = vars(get_args())
sys.argv = original_argv

model_args.update({
    'model': 'equivariant-transformer',
    'output_model': 'Scalar',
    'derivative': True,
    'embedding_dimension': 64,
    'num_layers': 2,
    'max_num_neighbors': 128,
})

print("\nInitializing TorchMD-Net Equivariant Transformer...")
model = create_model(model_args).to(device)

print(f"\nRunning Forward Pass with {num_atoms} atoms...")
try:
    pred_y, pred_neg_dy = model(batch.z, batch.pos, batch=batch.batch)
    print("✅ FORWARD PASS SUCCESSFUL!")
    print(f"Predicted Energy (Scalar): {pred_y.shape}")
    print(f"Predicted Forces (dY/dPos): {pred_neg_dy.shape}")
except Exception as e:
    print(f"❌ FORWARD PASS FAILED: {e}")
